# 00 — Data Cleaning & Customer-Level Table

Loads the raw Online Retail II invoice lines, applies the quality treatments from
brief section 4.1, and builds the customer-level feature/target table (section 1.3)
using a 9 September 2011 cutoff to avoid data leakage.

Outputs `data/processed/invoice_lines_clean.csv` and `data/processed/customer_table.csv`
for use by every later notebook.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.cleaning import load_raw, clean_invoice_lines
from src.features import build_customer_table, CUTOFF_DATE

pd.set_option('display.max_columns', None)

In [2]:
raw = load_raw()
print(raw.shape)
raw.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
clean = clean_invoice_lines(raw)
print(clean.shape)
print('Missing CustomerID share:', clean['CustomerID'].isna().mean())
print('Cancellation share:', clean['IsCancellation'].mean())
clean.head()

(1021534, 10)
Missing CustomerID share: 0.22248402892121066
Cancellation share: 0.017683209761006485


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,IsCancellation,LineRevenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0


In [4]:
n = len(raw)
inv = raw['Invoice'].astype(str)
codes = raw['StockCode'].astype(str).str.upper()

is_cancel = inv.str.startswith('C')

audit = pd.DataFrame({
    'Issue': [
        'Missing CustomerID',
        'Cancellation invoices (start with C)',
        'Negative quantity, not a cancellation',
        'Price <= 0',
        'Exact duplicate rows',
    ],
    'Rows': [
        raw['CustomerID'].isna().sum(),
        is_cancel.sum(),
        ((raw['Quantity'] < 0) & ~is_cancel).sum(),
        (raw['Price'] <= 0).sum(),
        raw.duplicated().sum(),
    ],
})
audit['% of rows'] = (audit['Rows'] / n * 100).round(2)
print('Total raw rows:', n)
audit

Total raw rows: 1067371


,Issue,Rows,% of rows
0,Missing CustomerID,243007,22.77
1,Cancellation invoices (start with C),19494,1.83
2,"Negative quantity, not a cancellation",3457,0.32
3,Price <= 0,6207,0.58
4,Exact duplicate rows,34335,3.22


In [5]:
odd_codes = codes[~codes.str.match(r'^\d{5}')]
odd_codes.value_counts().head(30)

StockCode
POST            2122
DOT             1446
M               1426
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
GIFT_0001_20      29
GIFT_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
GIFT_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14
GIFT_0001_50       8
GIFT_0001_40       7
DCGS0069           6
B                  6
DCGS0004           5
GIFT_0001_80       4
DCGS0072           4
DCGS0066N          4
DCGS0068           3
GIFT_0001_70       3
Name: count, dtype: int64

In [6]:
raw[['Quantity', 'Price']].describe()

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04


In [7]:
raw.nlargest(10, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'CustomerID']]

,Invoice,StockCode,Description,Quantity,Price,CustomerID
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446.0
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902.0
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902.0
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902.0
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902.0
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,13256.0
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902.0
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,NaN
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940.0


In [8]:
clean.to_csv('../data/processed/invoice_lines_clean.csv', index=False)

In [9]:
customer_table = build_customer_table(clean, cutoff=CUTOFF_DATE)
print(customer_table.shape)
customer_table.describe()

(5262, 12)


,CustomerID,Recency,Frequency,Monetary,DistinctProducts,TenureDays,AvgBasketValue,IsUK,AcquiredInQ4,CancellationRate,FutureSpend,Repurchase
count,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000,5262.000000
mean,15326.435766,206.346826,5.702585,2597.962478,74.353478,430.396807,365.726348,0.911821,0.328582,0.115058,529.437841,0.432915
std,1712.877317,174.502526,11.229489,12090.752679,104.811655,180.400000,539.305208,0.283583,0.469742,0.166267,3145.077531,0.495526
min,12346.000000,0.000000,1.000000,2.900000,1.000000,0.000000,2.900000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,13851.250000,49.000000,1.000000,315.722500,18.000000,310.000000,173.778750,1.000000,0.000000,0.000000,0.000000,0.000000
50%,15316.500000,163.000000,3.000000,775.480000,41.000000,472.000000,274.915833,1.000000,0.000000,0.000000,0.000000,0.000000
75%,16809.750000,324.000000,6.000000,2060.465000,91.000000,585.000000,407.466023,1.000000,1.000000,0.211722,429.435000,1.000000
max,18287.000000,646.000000,287.000000,456780.490000,2179.000000,646.000000,14844.766667,1.000000,1.000000,0.800000,112721.010000,1.000000


In [10]:
customer_table['Repurchase'].mean()

np.float64(0.43291524135309767)

In [11]:
customer_table.to_csv('../data/processed/customer_table.csv', index=False)